It creates:

    - ReadmeFiles\owner@repo.md

    - Manifests\owner@repo.csv (paths)

    - Metadata\owner@repo.json

    - Outputs\fetched_index.csv

Then read them for removal keywords


In [ ]:
# === stage1_fetch_data.py ===
import os
import requests
import pandas as pd
from base64 import b64decode
from dotenv import load_dotenv
from time import sleep

# === Load GitHub tokens ===
load_dotenv("All_Tokens.env")
tokens = [os.getenv(f"GITHUB_TOKEN_{i}") for i in range(1, 7)]
tokens = [t for t in tokens if t]
token_index = 0
if not tokens:
    raise ValueError("❌ No GitHub tokens found in All_Tokens.env")

def get_headers():
    global token_index
    token = tokens[token_index]
    token_index = (token_index + 1) % len(tokens)
    return {"Authorization": f"token {token}"}

# === Paths ===
base_dir = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14"
input_path = os.path.join(base_dir, "step2_manifest_final_output.csv")
readme_dir = os.path.join(base_dir, "ReadmeFiles")
manifest_dir = os.path.join(base_dir, "ManifestPaths")
os.makedirs(readme_dir, exist_ok=True)
os.makedirs(manifest_dir, exist_ok=True)

# === Load Data ===
df = pd.read_csv(input_path)
repos = df["html_url"].dropna().apply(lambda x: '/'.join(x.strip('/').split('/')[-2:]))

# === Fetch loop ===
for i, repo in enumerate(repos, 1):
    print(f"🔍 [{i}/{len(repos)}] Fetching: {repo}")
    try:
        # Fetch README
        r_readme = requests.get(f"https://api.github.com/repos/{repo}/readme", headers=get_headers())
        if r_readme.status_code == 200:
            content = r_readme.json().get("content", "")
            readme_text = b64decode(content).decode('utf-8', errors='ignore')
        else:
            readme_text = ""
        with open(os.path.join(readme_dir, repo.replace("/", "@") + ".md"), "w", encoding="utf-8") as f:
            f.write(readme_text)

        # Fetch Manifest Paths
        r_manifest = requests.get(f"https://api.github.com/search/code?q=filename:AndroidManifest.xml+repo:{repo}", headers=get_headers())
        paths = [item["path"] for item in r_manifest.json().get("items", [])] if r_manifest.status_code == 200 else []
        with open(os.path.join(manifest_dir, repo.replace("/", "@") + ".txt"), "w", encoding="utf-8") as f:
            f.write("\n".join(paths))

    except Exception as e:
        print(f"❌ Error with {repo}: {e}")
        sleep(1)

print("\n✅ Fetch complete.")


🔍 Reviewing 28250 repos...

💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14\step3_removal_keyword_output_interim.csv
💾 Interim saved to: C:\Android Mob

Changes to the pervious version v2.0 t0 make V3.0:

Since these changes are to the published paper, are not used in this paper becuase we want to refer to the published code

- Upgraded manifest check to exclude auto-generated path from removal keyword checking: build | intermediates | generated
- Upgreade the regex for analyzing the readme content to not to skip keywords followed by a punctuation or "."


In [ ]:
import os
import re
import pandas as pd

# === Base keywords for all checks ===
base_keywords = [
    'example', 'sample', 'demo', 'test', 'debug', 'presentation', 'module', 'components',
    'lib', 'library', 'sdk', 'utils', 'utility', 'plugin', 'widget', 'playground', 'framework',
    'architecture', 'skeleton', 'collection', 'starting point', 'protocol', 'benchmark', 'hackathon',
    'classroom', 'course', 'exercise', 'assignment', 'homework', 'assessment', 'interview', 'asset',
    'template', 'catalog', 'tutorial'
]

# === Keyword separation by target ===
manifest_keywords = base_keywords.copy()  # 'tool' not included here
non_manifest_keywords = base_keywords + ['tool']  # 'tool' added for repo name, description, README

preceded_by = ['this', 'is a', 'is an', 'our', 'my']
not_preceded_by = ['using', 'with']

def has_removal_context(text, k):
    if re.search(rf'(?<!\S){re.escape(k)}(?=\W|$)', text, re.IGNORECASE):
        if any(re.search(rf'(?<!\S){re.escape(w)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?=\W|$)', text, re.IGNORECASE) for w in preceded_by):
            if not any(re.search(rf'{re.escape(n)}\s+(\S+\s+){{0,4}}{re.escape(k)}(?=\W|$)', text, re.IGNORECASE) for n in not_preceded_by):
                return True
    return False

# === Paths ===
base_dir = r"C:\Android Mobile App\Step1_URL_Search\Type_1_Searching_Pipeline_July 14"
input_path = os.path.join(base_dir, "step2_manifest_final_output.csv")
output_path = os.path.join(base_dir, "step3_removal_keyword_output.csv")
readme_dir = os.path.join(base_dir, "ReadmeFiles")
manifest_dir = os.path.join(base_dir, "ManifestPaths")

# === Load dataset ===
df = pd.read_csv(input_path)
for col in ["removal_keyword_flag", "removal_reason", "Valid_Repo_Step3"]:
    if col not in df.columns:
        df[col] = "none"

# === Analyze each repo ===
for idx, row in df.iterrows():
    if row["Valid_Repo_Step2"] != "yes" or row["Valid_Repo_Step3"] != "none":
        continue

    repo = '/'.join(row["html_url"].strip('/').split('/')[-2:])
    name = row["name"].lower() if pd.notna(row["name"]) else ""
    topics = str(row.get("topics", "")).lower()
    description = str(row.get("description", "")).lower()

    readme_path = os.path.join(readme_dir, repo.replace("/", "@") + ".md")
    manifest_path = os.path.join(manifest_dir, repo.replace("/", "@") + ".txt")
    readme = open(readme_path, encoding="utf-8").read() if os.path.exists(readme_path) else ""
    manifest_paths = open(manifest_path, encoding="utf-8").read().splitlines() if os.path.exists(manifest_path) else []

    # === Remove build-related manifest paths ===
    manifest_paths = [
        p for p in manifest_paths
        if not re.search(r'\b(build|intermediates|generated|outputs)\b', p.lower())
    ]

    reasons = []

    # === Manifest path match (without 'tool') ===
    if any(k in path.lower() for k in manifest_keywords for path in manifest_paths):
        reasons.append("manifest_path")

    # === Repo name / topic match (with 'tool') ===
    if any(k in name for k in non_manifest_keywords) or any(k in topics for k in non_manifest_keywords):
        reasons.append("repo_name")

    # === Description contextual match ===
    if any(has_removal_context(description, k) for k in non_manifest_keywords):
        reasons.append("description")

    # === README contextual match ===
    if any(has_removal_context(readme, k) for k in non_manifest_keywords):
        reasons.append("readme")

    # === Save result ===
    flag = "yes" if reasons else "no"
    df.at[idx, "removal_keyword_flag"] = flag
    df.at[idx, "Valid_Repo_Step3"] = "no" if flag == "yes" else "yes"
    df.at[idx, "removal_reason"] = ", ".join(reasons) if reasons else "none"

# === Save results ===
df.to_csv(output_path, index=False)
print(f"✅ Analysis complete. Saved to: {output_path}")
